Code extracted from nems/scripts/demo_script_2p.py

In [ ]:
%matplotlib inline
import os
import logging
import pandas as pd
import pickle
import sys
import numpy as np
import matplotlib.pyplot as plt

from nems0.gui.recording_browser import browse_recording, browse_context
import nems
import nems0.initializers
import nems0.priors
import nems0.preprocessing as preproc
import nems0.plots.api as nplt
import nems0.modelspec as ms
import nems0.xforms as xforms
import nems0.utils
import nems0.uri
import nems0.recording as recording
from nems0.signal import RasterizedSignal
from nems0.fitters.api import scipy_minimize
import nems0.db as nd

log = logging.getLogger(__name__)


## PARAMETERS AND FILE LOCATIONS ##

In [ ]:
# data file and results locations
# defined in nems/nems/configs/settings.py, which will override
# defaults in nems/nems/configs/defaults.py
results_dir = nems.get_setting('NEMS_RESULTS_DIR')
recordings_dir = nems.get_setting('NEMS_RECORDINGS_DIR')

# 2p data from Polley Lab at EPL
respfile = os.path.join(recordings_dir, 'data_nems_2p/neurons.csv')
stimfile = os.path.join(recordings_dir, 'data_nems_2p/stim_spectrogram.csv')
exptid = "POL001"
cellid = "POL001-080"
batch = 1  # define the group of data this belong to (eg, 1: A1, 2: AAF, etc)
load_command='nems.demo.loaders.load_polley_data'


Alternatively, this is a more complex model (slower but possibly more accurate):

In [ ]:
modelspecname = 'dlog-wc.9x2.g-fir.2x15-lvl.1'

This one includes short-term plasticity (STP) for each spectral channel. Again, probably not helpful for such a sparse stimulus

In [ ]:
modelspecname = 'dlog-wcg.18x1-stp.1-fir.1x15-lvl.1-dexp.1'

This model schema is simple and fast

In [ ]:
modelspecname = 'wc.9x1.g-fir.1x15-lvl.1'

Specialized function to load 2p data (may be slow!)

In [ ]:
from nems0.demo.loaders import load_polley_data
r = load_polley_data(load_command=load_command,
                respfile=respfile,
                stimfile=stimfile,
                exptid=exptid,
                cellid=cellid)
print(r.keys())
print(r['stim'].shape)

## DO THE FIT ##

Define the `xforms` script for loading, fitting, and assessing performance

In [ ]:
# metadata describing the experiment that will be logged in the database if the model is saved
meta = {'cellid': cellid, 'exptid': exptid, 'batch': batch, 'modelname': modelspecname}

xfspec = []
xfspec.append(['nems.xforms.load_recording_wrapper',
               {'load_command': load_command,
                'respfile': respfile,
                'stimfile': stimfile,
                'exptid': exptid,
                'cellid': cellid}])

# reserve 10% of the data for validation
xfspec.append(['nems.xforms.split_at_time', {'valfrac': 0.1}])

xfspec.append(['nems.xforms.init_from_keywords',
               {'keywordstring': modelspecname, 'meta': meta}])

xfspec.append(['nems.xforms.fit_basic_init', {}])
xfspec.append(['nems.xforms.fit_basic', {}])
xfspec.append(['nems.xforms.predict',    {}])
# xfspec.append(['nems.xforms.add_summary_statistics',    {}])
xfspec.append(['nems.analysis.api.standard_correlation', {},
               ['est', 'val', 'modelspec', 'rec'], ['modelspec']])
xfspec.append(['nems.xforms.plot_summary',    {}])


First `xforms` command calls the loading function

In [ ]:
xfspec[0]

Here's the command that defines the `modelspec`

In [ ]:
xfspec[2]

actually do the fit

In [ ]:
ctx, log_xf = xforms.evaluate(xfspec)

# shorthand for iterating through each xfspec entry:
# ctx = {}
# for xfa in xfspec:
#     ctx = xforms.evaluate_step(xfa, ctx)


## LOOK AT THE RESULTS ##

In [ ]:
ctx['val'].signals

In [ ]:
modelspec = ctx['modelspec']
modelspec[0]['plot_fns']

In [ ]:
modelspec = ctx['modelspec']
modelspec[0]['plot_fns'].append('nems.plots.api.spectrogram')
modelspec[0]['plot_fn_idx']=2

In [ ]:
# testing new plotting system
modelspec[0]['plot_fn_idx']=2
fig=modelspec.quickplot(ctx['val'])


In [ ]:
modelspec.meta

## SAVE YOUR RESULTS ##

In [ ]:
# save results to file
destination = os.path.join(results_dir, str(batch), xforms.get_meta(ctx)['cellid'],
                           ms.get_modelspec_longname(ctx['modelspec']))
log.info('Saving modelspec(s) to {0} ...'.format(destination))
xforms.save_analysis(destination,
                     recording=ctx['rec'],
                     modelspec=ctx['modelspec'],
                     xfspec=xfspec,
                     figures=ctx['figures'],
                     log=log_xf)

# save summary of results to a database
log.info('Saving metadata to db  ...')
modelspec = ctx['modelspec']
modelspec.meta['modelpath'] = destination
modelspec.meta['figurefile'] = destination + 'figure.0000.png'
nd.update_results_table(modelspec)
    

## RELOAD ##

Load a model that has previously been saved. Specified uniquely by cellid/batch/modelspecname

In [ ]:
# reload using:
# repeated from above
import nems0.db as nd
import nems0.xforms as xforms

# pick your cell/batch/model
cellid = "POL001-080"
batch = 1  # define the group of data this belong to (eg, 1: A1, 2: AAF, etc)
modelspecname = 'wc.9x1.g-fir.1x15-lvl.1'

# find the results in the database
d=nd.get_results_file(batch=batch, cellids=[cellid], modelnames=[modelspecname])
d.loc[0]
filepath = d['modelpath'][0] + '/'

# load and display results
xfspec, ctx = xforms.load_analysis(filepath, eval_model=True)


Query the database for all models fit for a single cell/batch:

In [ ]:
import nems0.db as nd
import nems0.xforms as xforms

# pick your cell/batch/model
cellid = "POL001-080"
batch = 1  # define the group of data this belong to (eg, 1: A1, 2: AAF, etc)

# find the results in the database
d=nd.get_results_file(batch=batch, cellids=[cellid])
d[['modelname','r_test','figurefile']]

In [ ]:
modelspec=ctx['modelspec']
modelspec.meta